Transform our posts collection to prepare it for the model: 
1- unset the shortcode field 
2- group posts by owner id
3- concatenate captions
4- export data to a new collection

In [ ]:
from shared.config import get_mongo_config
from shared.mongo import MongoConnection

db = MongoConnection().get_database()
mongo_config = get_mongo_config()
posts_collection = mongo_config.get("POSTS_COLLECTION", "posts_merged")
db_name = mongo_config.get("DB_NAME", "InfluencersMarketing")

result = db[posts_collection].aggregate(
    [
        {"$unset": "shortcode"},
        {"$group": {"_id": "$owner.id", "post_ids": {"$push": "$$ROOT.id"}, "posts": {"$push": "$$ROOT.caption"}}},
        {
            "$set": {
                "posts": {"$reduce": {"input": "$posts", "initialValue": "", "in": {"$concat": ["$$value", "$$this"]}}}
            }
        },
        {"$out": {"db": db_name, "coll": "Posts"}},
    ]
)